# Bước 00: Hotfix Join Thời Tiết Causal — Notebook Kiểm Chứng Độc Lập
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

## 1. TỔNG QUAN VÀ MỤC TIÊU

Notebook này **tách riêng** đoạn hotfix join thời tiết causal (vốn nằm trong
`01_reindex_mask_outlier.ipynb`, bước 3.1) ra một file độc lập, để kiểm chứng/trình bày
tách biệt mà **không đụng vào pipeline chính đang chạy** (01→06).

**An toàn tuyệt đối với pipeline đang chạy:** notebook này chỉ ĐỌC
`data/mlmart_base/v3_preprocessing.parquet` (input gốc, không phải output của bất kỳ bước
nào trong 01-06), và GHI ra thư mục riêng biệt hoàn toàn
`data/model/v3/00_hotfix_audit/` — không trùng, không ghi đè bất kỳ file nào pipeline
chính (01→06) đang đọc hoặc ghi.

**Bối cảnh bug:** trước hotfix (2026-07-30), một số dòng bị join với `weather_timestamp`
**SAU** `timestamp` của chính nó — tức là dùng dữ liệu thời tiết CHƯA XẢY RA tại thời điểm
dự báo (leakage thời gian). Hotfix join lại theo quy tắc causal: mỗi dòng chỉ được lấy thời
tiết của khung giờ đã trôi qua (`floor` về đầu giờ), không bao giờ lấy giờ tương lai.

## 2. Import thư viện và khai báo tham số

In [ ]:
# ── Căn chỉnh lại thời tiết Causal (Code NGUYÊN BẢN từ srcs/00_utils/04_realign_mlmart_weather.py) ──
WEATHER_COLUMNS = (
    'weather_id', 'weather_type_id', 'weather_timestamp', 'weather_is_day',
    'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation',
    'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid',
    'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration',
    'weather_code', 'weather_type_is_day', 'weather_condition', 'weather_description'
)
LOOKUP_KEY = ("site_id", "_weather_hour")

# 1. Trích xuất bảng tra thời tiết chuẩn tại minute 00 (Hàm load_hourly_lookup)
_frame_m0 = df[pd.to_datetime(df['timestamp']).dt.minute.eq(0)].copy()
_frame_m0['_weather_hour'] = pd.to_datetime(_frame_m0['timestamp'], errors='raise')
_lookup = _frame_m0[[*LOOKUP_KEY, *[c for c in WEATHER_COLUMNS if c in df.columns]]].sort_values([*LOOKUP_KEY, 'weather_id'], kind='stable')
_lookup = _lookup.drop_duplicates(list(LOOKUP_KEY), keep='first').set_index(list(LOOKUP_KEY))

# 2. Realign thời tiết theo mốc timestamp.dt.floor('h') (Hàm realign_batch)
_timestamp = pd.to_datetime(df['timestamp'], errors='raise')
_keys = pd.MultiIndex.from_arrays(
    [df['site_id'].to_numpy(), _timestamp.dt.floor('h').to_numpy()],
    names=LOOKUP_KEY
)
_aligned = _lookup.reindex(_keys).reset_index(drop=True)

for col in WEATHER_COLUMNS:
    if col in df.columns:
        df[col] = _aligned[col].to_numpy()

df['weather_timestamp'] = _timestamp.dt.floor('h')

del _frame_m0, _lookup, _keys, _aligned
gc.collect()


## 3. Đọc dữ liệu gốc

In [ ]:
df = pd.read_parquet(INPUT_PATH)
print(f"Đang đọc dữ liệu từ: {INPUT_PATH}")
print(f"Tổng số dòng: {len(df)}")
print(f"Số site: {df[SITE_COL].nunique()}")
display(df.head(3))

## 4. Đo lường TRƯỚC khi sửa — bằng chứng bug leakage thời gian

In [ ]:
# Cell preserved


### Nhận xét
Nếu `_leak_truoc > 0`, nghĩa là dữ liệu gốc vẫn còn dòng dùng thời tiết chưa xảy ra tại
thời điểm dự báo — đây chính xác là bug leakage thời gian gây trễ pha đã fix ngày
2026-07-30. Nếu `_leak_truoc == 0`, nghĩa là bug đã được xử lý ở nguồn
(`v3_preprocessing.parquet`) từ trước, và bước dưới đây chạy như lưới an toàn (idempotent).

## 5. Áp dụng hotfix — join lại thời tiết theo đúng quy tắc causal

In [ ]:
# Cell preserved


## 6. Đo lường SAU khi sửa — xác nhận hết leakage

In [ ]:
# ── Căn chỉnh lại thời tiết Causal (Code NGUYÊN BẢN từ srcs/00_utils/04_realign_mlmart_weather.py) ──
WEATHER_COLUMNS = (
    'weather_id', 'weather_type_id', 'weather_timestamp', 'weather_is_day',
    'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation',
    'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid',
    'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration',
    'weather_code', 'weather_type_is_day', 'weather_condition', 'weather_description'
)
LOOKUP_KEY = ("site_id", "_weather_hour")

# 1. Trích xuất bảng tra thời tiết chuẩn tại minute 00 (Hàm load_hourly_lookup)
_frame_m0 = df[pd.to_datetime(df['timestamp']).dt.minute.eq(0)].copy()
_frame_m0['_weather_hour'] = pd.to_datetime(_frame_m0['timestamp'], errors='raise')
_lookup = _frame_m0[[*LOOKUP_KEY, *[c for c in WEATHER_COLUMNS if c in df.columns]]].sort_values([*LOOKUP_KEY, 'weather_id'], kind='stable')
_lookup = _lookup.drop_duplicates(list(LOOKUP_KEY), keep='first').set_index(list(LOOKUP_KEY))

# 2. Realign thời tiết theo mốc timestamp.dt.floor('h') (Hàm realign_batch)
_timestamp = pd.to_datetime(df['timestamp'], errors='raise')
_keys = pd.MultiIndex.from_arrays(
    [df['site_id'].to_numpy(), _timestamp.dt.floor('h').to_numpy()],
    names=LOOKUP_KEY
)
_aligned = _lookup.reindex(_keys).reset_index(drop=True)

for col in WEATHER_COLUMNS:
    if col in df.columns:
        df[col] = _aligned[col].to_numpy()

df['weather_timestamp'] = _timestamp.dt.floor('h')

del _frame_m0, _lookup, _keys, _aligned
gc.collect()


## 7. Cổng kiểm tra — dừng nếu vẫn còn leakage

In [ ]:
# Cell preserved


## 8. Export Processed Dataset — ghi ra thư mục audit riêng

In [ ]:
df.to_parquet(OUTPUT_PATH, index=False)
print("--- HOÀN TẤT ---")
print(f"Đã ghi file kiểm chứng ra: {OUTPUT_PATH}")
print(f"Shape cuối cùng: {df.shape[0]} dòng x {df.shape[1]} cột")
print("\nLưu ý: file này CHỈ dùng để kiểm chứng/trình bày, KHÔNG được pipeline chính")
print("(01->06) đọc hay ghi đè. Không ảnh hưởng tới lần train đang chạy.")